In [3]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
from collections import deque
import os
import random

class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi.
    Mendukung pemilihan acak untuk training, dan pemilihan spesifik untuk evaluasi.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8 # MTU standar
        
        if os.path.exists(folder_path):
            # Ambil file dan urutkan agar evaluasi 1 sampai 8 berurutan
            files = [f for f in os.listdir(folder_path) if f.endswith('.txt') or f.endswith('.log')]
            files.sort() 
            
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                                
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            throughput_mbps = [max(0.1, tp) for tp in throughput_mbps]
                            
                            self.traces.append({
                                "name": file,
                                "data": throughput_mbps
                            })
                except Exception as e:
                    print(f"⚠️ Gagal memproses {file}: {e}")
            
            if self.traces:
                print(f"✅ Berhasil memuat {len(self.traces)} file trace (Terurut).")
        
        if not self.traces:
            print("⚠️ Folder kosong. Menggunakan pola sintetis...")
            self.traces.append({"name": "synth", "data": np.clip(10 + 5 * np.sin(np.linspace(0, 50, 1000)), 0.5, 20).tolist()})

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        """Untuk mode Training: Acak file dan acak titik mulai"""
        self.active_trace = random.choice(self.traces)
        if len(self.active_trace["data"]) > 105:
            self.ptr = random.randint(0, len(self.active_trace["data"]) - 105)
        else:
            self.ptr = 0
        return self.active_trace["name"]

    def set_trace_index(self, index):
        """Untuk mode Evaluasi: Set file secara spesifik dari awal"""
        self.active_trace = self.traces[index % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        if not self.active_trace:
            self.select_random_trace()
        
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class TandonMahimahiEnv(gym.Env):
    def __init__(self, trace_manager):
        super(TandonMahimahiEnv, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0] # Low, Mid, High
        
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0, -10, -20, 0, 0]),
            high=np.array([30, 50, 2, 10, 20, 1000, 100]),
            dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)
        self.state = None
        self.max_steps = 100
        self.current_step = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Cek apakah ini mode evaluasi spesifik
        if options and "trace_index" in options:
            trace_name = self.trace_manager.set_trace_index(options["trace_index"])
        else:
            trace_name = self.trace_manager.select_random_trace()
            
        initial_tp = self.trace_manager.get_next_bandwidth()
        self.state = np.array([15.0, initial_tp, 1.0, 0.0, 0.0, 40.0, 0.0], dtype=np.float32)
        self.current_step = 0
        return self.state, {"trace": trace_name}

    def step(self, action):
        buffer, last_tp_avg, last_action, _, _, rtt, dropped = self.state
        chosen_bitrate = self.bitrates[action]
        raw_tp = self.trace_manager.get_next_bandwidth()
        
        seg_duration = 5.0
        download_time = (chosen_bitrate * seg_duration / (raw_tp + 0.1)) + (rtt / 1000.0)
        stalling = max(0, download_time - buffer)
        
        new_buffer = max(0, buffer - download_time) + seg_duration
        new_buffer = min(new_buffer, 30.0)
        
        buf_trend = np.clip(new_buffer - buffer, -10, 10)
        tp_trend = np.clip(raw_tp - last_tp_avg, -20, 20)
        
        # --- REWARD LOGIC (TWEAKED UNTUK MENGURANGI FLATNESS) ---
        # 1. Hadiah memilih kualitas dinaikkan agar AI lebih tergiur mencoba High
        reward = chosen_bitrate * 2.5 
        
        # 2. Penalti gonta-ganti sedikit diturunkan agar lebih lincah
        reward -= abs(action - last_action) * 0.5 
        
        # 3. Penalti macet tetap berat (Safety First)
        if stalling > 0:
            reward -= 150.0 
        
        # 4. Zona darurat tidak terlalu menakutkan kecuali nekat ambil High
        if new_buffer < 10.0:
            reward -= 10.0 
            if action == 2: 
                reward -= 40.0 
            
        # 5. Penalti keras jika pelit padahal internet bagus dan tandon penuh
        if action == 0 and raw_tp > 8.0 and new_buffer > 20.0:
            reward -= 25.0 

        self.state = np.array([new_buffer, raw_tp, float(action), buf_trend, tp_trend, 40.0, 0.0], dtype=np.float32)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self.state, reward, done, False, {"raw_tp": raw_tp}

def save_plots(log_folder, eval_histories):
    # Plot Learning Curve
    try:
        x, y = ts2xy(load_results(log_folder), 'timesteps')
        y_df = pd.Series(y)
        y_smooth = y_df.rolling(window=50, min_periods=1).mean()

        plt.figure(figsize=(10, 5))
        plt.plot(x, y, alpha=0.3, color='gray', label='Reward per Episode')
        plt.plot(x, y_smooth, color='blue', linewidth=2, label='Rata-rata Reward')
        plt.xlabel('Timesteps')
        plt.ylabel('Skor Reward')
        plt.title("Track Record Pembelajaran Agen RL")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(log_folder, "learning_curve.png"))
        plt.close()
    except Exception as e:
        print(f"⚠️ Gagal menyimpan Learning Curve: {e}")

    # Plot ke-8 Trace Evaluasi
    for name, df in eval_histories:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        
        # Normalisasi visual AI Decision agar sejajar dengan grafik Mbps
        # Action 0 -> 2 Mbps, Action 1 -> 5 Mbps, Action 2 -> 10 Mbps (Visual saja)
        visual_action = df['Action'] * 4 + 2 
        
        ax1.plot(df.index, df['Throughput'], label=f'Internet Asli ({name})', color='blue', alpha=0.5)
        ax1.step(df.index, visual_action, label='Keputusan Bitrate AI', color='red', where='post', linewidth=2)
        ax1.set_title(f"Evaluasi Agen pada {name}")
        ax1.set_ylabel("Mbps / Kualitas")
        ax1.legend()
        ax1.grid(True, alpha=0.2)
        
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.2, label='Isi Tandon (Buffer)')
        ax2.axhline(y=10, color='orange', linestyle='--', label='Batas Darurat')
        ax2.set_ylabel("Detik")
        ax2.set_xlabel("Segmen Video")
        ax2.set_ylim(0, 35)
        ax2.legend()
        ax2.grid(True, alpha=0.2)
        
        plt.tight_layout()
        plt.savefig(os.path.join(log_folder, f"eval_result_{name}.png"))
        plt.close()

def run_multi_trace_training():
    print("🎬 Memulai Pelatihan RL dengan Konversi Mahimahi Asli...")
    
    log_dir = "./rl_logs/"
    os.makedirs(log_dir, exist_ok=True)
    
    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi") 
    env = TandonMahimahiEnv(tm)
    env = Monitor(env, log_dir)
    
    # Entropi dipertahankan 0.02 agar agen rajin bereksperimen (tidak diam saja)
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.0003, 
                n_steps=2048, 
                batch_size=128,
                ent_coef=0.02)
    
    print("⏳ Sedang melatih model... (200,000 langkah)")
    model.learn(total_timesteps=200000)
    model.save("ndn_video_brain_tandon_multi_trace")
    print("✅ Model '.zip' telah diperbarui.")

    # --- EVALUASI PASTI UNTUK SEMUA TRACE ---
    eval_histories = []
    num_traces = len(tm.traces)
    print(f"\n--- Menjalankan Evaluasi Pasca-Latih untuk {num_traces} File ---")
    
    for i in range(num_traces):
        # Paksa environment menggunakan index file tertentu
        obs, info = env.reset(options={"trace_index": i})
        name = info["trace"]
        history = []
        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            history.append({
                'Throughput': info_step['raw_tp'],
                'Buffer': obs[0],
                'Action': action
            })
        eval_histories.append((name, pd.DataFrame(history)))

    print("🖼️ Menyimpan 8 grafik ke folder ./rl_logs/ ...")
    save_plots(log_dir, eval_histories)
    print("🎉 Selesai! Silakan cek folder ./rl_logs/ Anda.")

if __name__ == "__main__":
    run_multi_trace_training()

🎬 Memulai Pelatihan RL dengan Konversi Mahimahi Asli...
✅ Berhasil memuat 8 file trace (Terurut).
Using cpu device
Wrapping the env in a DummyVecEnv.
⏳ Sedang melatih model... (200,000 langkah)
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -47.5    |
| time/              |          |
|    fps             | 5172     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 100       |
|    ep_rew_mean          | 141       |
| time/                   |           |
|    fps                  | 3989      |
|    iterations           | 2         |
|    time_elapsed         | 1         |
|    total_timesteps      | 4096      |
| train/                  |           |
|    approx_kl            | 0.0315403 |
|    clip_fraction        